# PA3 · Task 2 — SigLIP → MLP connector → Qwen3-0.6B VLM, three-stage curriculum, visual-grounding evaluation

**Pipeline (every stage is idempotent and resumable):**
1. Download COCO-2014 captions/instances and VQAv2 questions/annotations/complementary pairs.
2. Seeded (6304) selection of every split: 6,000×3 training captions, 96 dev captions, annotation-derived presence/counting pair pools (train) and 24/64 dev/test pairs, natural VQAv2 complementary pairs 1,536 / 32 / 96 with 640/256/640, 12/8/12, 36/24/36 yes-no/number/other. All dev + test images are val2014, unique and disjoint across sets; training uses train2014 only. All IDs are saved.
3. Download only the needed images, cache the 196 SigLIP patch features (after SigLIP's final LayerNorm, applied once) in FP16, then free the vision encoder.
4. Train: Stage 1 caption alignment (connector only, 2,250 updates) → Stage 2 visual instruction tuning (+LoRA r16 on q/k/v/o, 1,500 updates, 50% presence / 25% counting / 25% caption replay) → Stage 3 natural VQA (750 updates, 70/10/10/10). Dev evaluation at 0, every 500 updates and at 2,250 / 3,750 / 4,500.
5. Final test evaluation under correct / complementary-swap / mismatched / no-image inputs with pair diagnostics, reference NLL and 4,000× pair bootstrap CIs.
6. Figures, tables, qualitative aligned examples, resources. Optional: the 49-token (2×2-pooled) repeat (`RUN_COMPRESSED=True`).

**Approximate T4 time:** downloads + selection ≈ 10–15 min · image download + SigLIP feature caching ≈ 10–15 min · training (4,500 updates incl. 12 dev evals) ≈ 1.3–1.8 h · final evaluation ≈ 5–10 min · analysis ≈ 2 min → **≈ 2–2.5 h**. Optional 49-token run ≈ +0.6–0.9 h.

In [1]:
import os, sys, subprocess
if not os.environ.get("PA3_SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.56,<6", "accelerate>=0.33",
                    "pillow", "pandas", "matplotlib", "sentencepiece", "protobuf"], check=True)

In [2]:
import os, sys, json, math, time, re, gc, glob, shutil, random, zipfile, io
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/ATML_PA3/task2")
    SCRATCH = Path("/content/scratch_task2")
elif IN_KAGGLE:
    WORK = Path("/kaggle/working/task2"); SCRATCH = Path("/tmp/scratch_task2")
else:
    WORK = Path(os.environ.get("PA3_WORK", "./pa3_work/task2")).resolve(); SCRATCH = WORK / "scratch"

SEED = 6304
QWEN_ID, QWEN_REV = "Qwen/Qwen3-0.6B", "main"
SIGLIP_ID = "google/siglip-base-patch16-224"
GRID = 14                                  # 14×14 = 196 patches
N_CAP_TRAIN_IMG, CAPS_PER_TRAIN_IMG, N_CAP_DEV = 6000, 3, 96
VQA_TRAIN = {"yes/no": 640, "number": 256, "other": 640}
VQA_DEV = {"yes/no": 12, "number": 8, "other": 12}
VQA_TEST = {"yes/no": 36, "number": 24, "other": 36}
GROUND_DEV = {"presence": 12, "counting": 12}           # 24 pairs / 48 questions
GROUND_TEST = {"presence": 32, "counting": 32}          # 64 pairs / 128 questions
PRES_PAIRS_PER_CAT, COUNT_PAIRS_PER_CAT = 80, 80        # training pool caps per category
MIN_AGREE = 8
MICRO_BS, GRAD_ACC = 8, 2
EFF_BS = MICRO_BS * GRAD_ACC                             # 16
STAGES = [dict(name="stage1_caption", updates=2250, conn_lr=1e-3, lora_lr=None,
               mix={"caption": 1.0}),
          dict(name="stage2_instruct", updates=1500, conn_lr=1e-4, lora_lr=2e-5,
               mix={"presence": 0.50, "counting": 0.25, "caption": 0.25}),
          dict(name="stage3_vqa", updates=750, conn_lr=5e-5, lora_lr=1e-5,
               mix={"vqa": 0.70, "presence": 0.10, "counting": 0.10, "caption": 0.10})]
TOTAL_UPDATES = sum(s["updates"] for s in STAGES)        # 4500
STAGE_ENDS = list(np.cumsum([s["updates"] for s in STAGES]))   # 2250, 3750, 4500
DEV_EVAL_AT = sorted(set(range(0, TOTAL_UPDATES + 1, 500)) | set(STAGE_ENDS))
CKPT_EVERY = 250
WARMUP_FRAC, MIN_LR_FRAC, WEIGHT_DECAY, CLIP = 0.05, 0.10, 0.01, 1.0
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]
CAP_Q, CAP_CAPTION, CAP_ANS = 80, 64, 16
MAX_NEW_TOKENS, GEN_BATCH = 16, 8
N_BOOT = 4000
CAPTION_PROMPT = "Write a short caption for this image."
ANSWER_INSTR = "Answer the question using a single word or phrase."

RUN_DATA, RUN_FEATURES, RUN_TRAIN, RUN_FINAL_EVAL, RUN_ANALYSIS = True, True, True, True, True
RUN_COMPRESSED = False            # optional 49-token experiment (2×2 average pooling before the connector)
DOWNLOAD_WORKERS = 32

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RAW_DIR, IMG_DIR = SCRATCH / "raw", SCRATCH / "images"
DATA_DIR, FEAT_DIR, RUNS_DIR = WORK / "data", WORK / "features", WORK / "runs"
FIG_DIR, TAB_DIR = WORK / "figures", WORK / "tables"
for d in [WORK, SCRATCH, RAW_DIR, IMG_DIR, DATA_DIR, FEAT_DIR, RUNS_DIR, FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("WORK =", WORK, "| device =", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

WORK = /kaggle/working/task2 | device = cuda Tesla T4


In [3]:
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def jdump(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.replace(tmp, path)

def jload(path):
    with open(path) as f:
        return json.load(f)

def append_jsonl(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")

def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def gb(n):
    return round(n / 1024**3, 3)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def autocast():
    return torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", dtype=torch.float16,
                          enabled=(DEVICE == "cuda"))

def no_autocast():
    return torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=False)

def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    import urllib.request
    tmp = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(5):
        try:
            urllib.request.urlretrieve(url, tmp); os.replace(tmp, dest); return dest
        except Exception as e:
            print(f"  retry {attempt+1} {url}: {e}"); time.sleep(2 + 3 * attempt)
    raise RuntimeError(f"download failed: {url}")

# ---- official VQA answer normalisation (VQA eval script: punctuation, digits, articles, contractions) ----
_CONTRACTIONS = {"aint": "ain't", "arent": "aren't", "cant": "can't", "couldve": "could've", "couldnt": "couldn't",
    "couldn'tve": "couldn't've", "couldnt've": "couldn't've", "didnt": "didn't", "doesnt": "doesn't", "dont": "don't",
    "hadnt": "hadn't", "hadnt've": "hadn't've", "hadn'tve": "hadn't've", "hasnt": "hasn't", "havent": "haven't",
    "hed": "he'd", "hed've": "he'd've", "he'dve": "he'd've", "hes": "he's", "howd": "how'd", "howll": "how'll",
    "hows": "how's", "Id've": "I'd've", "I'dve": "I'd've", "Im": "I'm", "Ive": "I've", "isnt": "isn't", "itd": "it'd",
    "itd've": "it'd've", "it'dve": "it'd've", "itll": "it'll", "let's": "let's", "maam": "ma'am", "mightnt": "mightn't",
    "mightnt've": "mightn't've", "mightn'tve": "mightn't've", "mightve": "might've", "mustnt": "mustn't",
    "mustve": "must've", "neednt": "needn't", "notve": "not've", "oclock": "o'clock", "oughtnt": "oughtn't",
    "ow's'at": "'ow's'at", "'ows'at": "'ow's'at", "'ow'sat": "'ow's'at", "shant": "shan't", "shed've": "she'd've",
    "she'dve": "she'd've", "she's": "she's", "shouldve": "should've", "shouldnt": "shouldn't",
    "shouldnt've": "shouldn't've", "shouldn'tve": "shouldn't've", "somebody'd": "somebodyd",
    "somebodyd've": "somebody'd've", "somebody'dve": "somebody'd've", "somebodyll": "somebody'll",
    "somebodys": "somebody's", "someoned": "someone'd", "someoned've": "someone'd've", "someone'dve": "someone'd've",
    "someonell": "someone'll", "someones": "someone's", "somethingd": "something'd", "somethingd've": "something'd've",
    "something'dve": "something'd've", "somethingll": "something'll", "thats": "that's", "thered": "there'd",
    "thered've": "there'd've", "there'dve": "there'd've", "therere": "there're", "theres": "there's", "theyd": "they'd",
    "theyd've": "they'd've", "they'dve": "they'd've", "theyll": "they'll", "theyre": "they're", "theyve": "they've",
    "twas": "'twas", "wasnt": "wasn't", "wed've": "we'd've", "we'dve": "we'd've", "weve": "we've", "werent": "weren't",
    "whatll": "what'll", "whatre": "what're", "whats": "what's", "whatve": "what've", "whens": "when's",
    "whered": "where'd", "wheres": "where's", "whereve": "where've", "whod": "who'd", "whod've": "who'd've",
    "who'dve": "who'd've", "wholl": "who'll", "whos": "who's", "whove": "who've", "whyll": "why'll", "whyre": "why're",
    "whys": "why's", "wont": "won't", "wouldve": "would've", "wouldnt": "wouldn't", "wouldnt've": "wouldn't've",
    "wouldn'tve": "wouldn't've", "yall": "y'all", "yall'll": "y'all'll", "y'allll": "y'all'll",
    "yall'd've": "y'all'd've", "y'alld've": "y'all'd've", "y'all'dve": "y'all'd've", "youd": "you'd",
    "youd've": "you'd've", "you'dve": "you'd've", "youll": "you'll", "youre": "you're", "youve": "you've"}
_MANUAL_MAP = {"none": "0", "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6",
               "seven": "7", "eight": "8", "nine": "9", "ten": "10"}
_ARTICLES = {"a", "an", "the"}
_PERIOD_STRIP = re.compile(r"(?!<=\d)(\.)(?!\d)")
_COMMA_STRIP = re.compile(r"(\d)(,)(\d)")
_PUNCT = [";", r"/", "[", "]", '"', "{", "}", "(", ")", "=", "+", "\\", "_", "-", ">", "<", "@", "`", ",", "?", "!"]

from functools import lru_cache

@lru_cache(maxsize=None)
def vqa_normalize(s):
    s = (s or "").replace("\n", " ").replace("\t", " ").strip()
    out = s
    for p in _PUNCT:
        if (p + " " in s or " " + p in s) or (re.search(_COMMA_STRIP, s) is not None):
            out = out.replace(p, "")
        else:
            out = out.replace(p, " ")
    out = _PERIOD_STRIP.sub("", out)
    words = [_MANUAL_MAP.get(w, w) for w in out.lower().split()]
    words = [_CONTRACTIONS.get(w, w) for w in words if w not in _ARTICLES]
    return " ".join(words)

def vqa_soft_acc(pred_norm, refs_norm):
    """s(â) = 1/10 Σ_i min(1, 1/3 Σ_{j≠i} 1[â = a_j]) over the ten normalised human answers."""
    m = [r == pred_norm for r in refs_norm]
    tot = sum(m)
    return float(np.mean([min(1.0, (tot - mi) / 3.0) for mi in m]))

## 1 · Data download and seeded selection

In [4]:
COCO_ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2014.zip"
VQA_BASE = "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/"
VQA_ZIPS = ["v2_Questions_Train_mscoco.zip", "v2_Questions_Val_mscoco.zip", "v2_Annotations_Train_mscoco.zip",
            "v2_Annotations_Val_mscoco.zip", "v2_Complementary_Pairs_Train_mscoco.zip",
            "v2_Complementary_Pairs_Val_mscoco.zip"]
COCO_IMG_URL = "http://images.cocodataset.org/{split}/COCO_{split}_{id:012d}.jpg"

# object phrasing for the generated questions
_SING_OVERRIDE = {"person": "a person", "sports ball": "a ball", "mouse": "a computer mouse", "skis": "skis",
                  "scissors": "scissors", "broccoli": "broccoli", "tv": "a TV", "hair drier": "a hair drier"}
_PLUR_OVERRIDE = {"person": "people", "sheep": "sheep", "mouse": "computer mice", "knife": "knives", "bus": "buses",
                  "couch": "couches", "bench": "benches", "sandwich": "sandwiches", "toothbrush": "toothbrushes",
                  "wine glass": "wine glasses", "sports ball": "balls", "tv": "TVs", "skis": "pairs of skis",
                  "scissors": "pairs of scissors", "broccoli": "pieces of broccoli", "hair drier": "hair driers",
                  "giraffe": "giraffes", "skateboard": "skateboards", "surfboard": "surfboards"}
_NO_COUNT = {"skis", "scissors", "broccoli", "carrot", "banana", "book"}          # mass / bunch / pair nouns
_SYNONYMS = {"person": ["person", "people", "man", "men", "woman", "women", "boy", "girl", "child", "children", "kid",
                        "kids", "player", "guy", "lady", "persons", "baby", "skier", "surfer", "rider"],
             "dining table": ["table"], "tv": ["tv", "television"], "couch": ["sofa", "couch"],
             "cell phone": ["phone", "cellphone"], "motorcycle": ["motorbike", "motorcycle"],
             "airplane": ["plane", "jet", "aircraft", "airplane"], "bicycle": ["bike", "bicycle"],
             "sports ball": ["ball"], "cup": ["cup", "mug"], "potted plant": ["plant"], "laptop": ["laptop", "computer"]}

def sing(name):
    if name in _SING_OVERRIDE:
        return _SING_OVERRIDE[name]
    return ("an " if name[0] in "aeiou" else "a ") + name

def plur(name):
    return _PLUR_OVERRIDE.get(name, name + ("es" if name.endswith(("s", "x", "ch", "sh")) else "s"))

def mention_words(name):
    return set(_SYNONYMS.get(name, [])) | {name, plur(name), name.split()[-1], plur(name).split()[-1]}

def fetch_raw():
    for f in VQA_ZIPS:
        download(VQA_BASE + f, RAW_DIR / f)
    download(COCO_ANN_URL, RAW_DIR / "annotations_trainval2014.zip")

def _read_zip_json(zip_path, member_suffix):
    with zipfile.ZipFile(zip_path) as z:
        name = next(n for n in z.namelist() if n.endswith(member_suffix))
        with z.open(name) as f:
            return json.load(f)

def load_coco(split):
    zp = RAW_DIR / "annotations_trainval2014.zip"
    caps = _read_zip_json(zp, f"captions_{split}2014.json")
    inst = _read_zip_json(zp, f"instances_{split}2014.json")
    return caps, inst

def load_vqa(split):
    S = split.capitalize()
    q = _read_zip_json(RAW_DIR / f"v2_Questions_{S}_mscoco.zip", f"v2_OpenEnded_mscoco_{split}2014_questions.json")
    a = _read_zip_json(RAW_DIR / f"v2_Annotations_{S}_mscoco.zip", f"v2_mscoco_{split}2014_annotations.json")
    p = _read_zip_json(RAW_DIR / f"v2_Complementary_Pairs_{S}_mscoco.zip", f"v2_mscoco_{split}2014_complementary_pairs.json")
    return q, a, p

def image_stats(inst):
    """Per image: {cat_name: dict(n=noncrowd count, crowd=#crowd, min_frac, max_frac)}."""
    cats = {c["id"]: c["name"] for c in inst["categories"]}
    size = {im["id"]: im["width"] * im["height"] for im in inst["images"]}
    st = defaultdict(dict)
    for a in inst["annotations"]:
        name = cats[a["category_id"]]
        d = st[a["image_id"]].setdefault(name, dict(n=0, crowd=0, min_frac=1.0, max_frac=0.0))
        if a.get("iscrowd", 0):
            d["crowd"] += 1
        else:
            fr = a["area"] / size[a["image_id"]]
            d["n"] += 1; d["min_frac"] = min(d["min_frac"], fr); d["max_frac"] = max(d["max_frac"], fr)
    return st, sorted(set(cats.values())), sorted(size)

def captions_by_image(caps):
    out = defaultdict(list)
    for a in sorted(caps["annotations"], key=lambda a: a["id"]):
        out[a["image_id"]].append(a["caption"].strip())
    return out

def vqa_pair_candidates(split, tok):
    q, a, pairs = load_vqa(split)
    qtext = {x["question_id"]: x["question"] for x in q["questions"]}
    ann = {x["question_id"]: x for x in a["annotations"]}
    out = []
    for q1, q2 in pairs:
        if q1 not in ann or q2 not in ann:
            continue
        A1, A2 = ann[q1], ann[q2]
        if A1["image_id"] == A2["image_id"] or qtext[q1].strip() != qtext[q2].strip():
            continue
        if A1["answer_type"] != A2["answer_type"]:
            continue
        info = []
        for A in (A1, A2):
            raw = [x["answer"] for x in A["answers"]]
            norm = [vqa_normalize(r) for r in raw]
            cons, k = Counter(norm).most_common(1)[0]
            if k < MIN_AGREE or cons == "":
                break
            group_raw = Counter(r.strip() for r, n in zip(raw, norm) if n == cons)
            best = max(group_raw.values())
            target = min(r for r, c in group_raw.items() if c == best)
            info.append(dict(qid=A["question_id"], image_id=A["image_id"], consensus=cons, target=target,
                             refs=raw, agree=k))
        if len(info) < 2 or info[0]["consensus"] == info[1]["consensus"]:
            continue
        out.append(dict(pair_id=f"vqa_{split}_{min(q1, q2)}_{max(q1, q2)}", question=qtext[q1].strip(),
                        qtype=A1["answer_type"], split=split, members=info))
    out.sort(key=lambda d: d["pair_id"])
    return out

def build_selection():
    out = DATA_DIR / "selection.json"
    if out.exists():
        return jload(out)
    t0 = time.time()
    fetch_raw()
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(QWEN_ID, revision=QWEN_REV)
    rng = np.random.default_rng(SEED)
    used_val = set()
    sel = dict(seed=SEED)
    # ---------- natural VQAv2 pairs ----------
    tr_c, va_c = vqa_pair_candidates("train", tok), vqa_pair_candidates("val", tok)
    sel["vqa_candidate_counts"] = dict(train=dict(Counter(c["qtype"] for c in tr_c)), val=dict(Counter(c["qtype"] for c in va_c)))
    vqa_train = []
    for t, n in VQA_TRAIN.items():
        pool = [c for c in tr_c if c["qtype"] == t]
        idx = rng.choice(len(pool), n, replace=False)
        vqa_train += [pool[i] for i in sorted(idx)]
    def take_val(quota, pool):
        order = rng.permutation(len(pool)); got = []; cnt = Counter()
        for i in order:
            c = pool[i]
            ims = [m["image_id"] for m in c["members"]]
            if cnt[c["qtype"]] >= quota.get(c["qtype"], 0) or any(im in used_val for im in ims):
                continue
            got.append(c); cnt[c["qtype"]] += 1; used_val.update(ims)
            if all(cnt[k] >= v for k, v in quota.items()):
                break
        assert all(cnt[k] >= v for k, v in quota.items()), cnt
        return sorted(got, key=lambda d: d["pair_id"])
    vqa_test = take_val(VQA_TEST, va_c)
    vqa_dev = take_val(VQA_DEV, va_c)
    # ---------- COCO captions + annotation stats ----------
    caps_tr, inst_tr = load_coco("train"); caps_va, inst_va = load_coco("val")
    cbi_tr, cbi_va = captions_by_image(caps_tr), captions_by_image(caps_va)
    st_tr, catnames, ims_tr = image_stats(inst_tr); st_va, _, ims_va = image_stats(inst_va)
    del inst_tr, inst_va, caps_tr, caps_va
    # training caption images (train2014)
    elig = [i for i in ims_tr if len(cbi_tr.get(i, [])) >= CAPS_PER_TRAIN_IMG]
    cap_imgs = sorted(rng.choice(elig, N_CAP_TRAIN_IMG, replace=False).tolist())
    cap_train = []
    for im in cap_imgs:
        cs = cbi_tr[im]
        for j in sorted(rng.choice(len(cs), CAPS_PER_TRAIN_IMG, replace=False).tolist()):
            cap_train.append(dict(image=f"train2014/{im}", caption=cs[j]))
    # ---------- annotation-derived grounding ----------
    _capcache = {}
    def mentions(cbi, im, name):
        key = (id(cbi), im)
        if key not in _capcache:
            phr = " ".join(cbi.get(im, [])).lower()
            _capcache[key] = (set(re.findall(r"[a-z]+", phr)), phr)
        words, phr = _capcache[key]
        return any((w in words) if " " not in w else (w in phr) for w in mention_words(name))
    def pres_pos(st, im, c):
        d = st.get(im, {}).get(c); return d is not None and d["n"] >= 1 and d["max_frac"] >= 0.01
    def pres_neg(st, cbi, im, c):
        return c not in st.get(im, {}) and not mentions(cbi, im, c)
    def count_of(st, im, c):
        d = st.get(im, {}).get(c)
        if d is None or c in _NO_COUNT or d["crowd"] > 0 or not (1 <= d["n"] <= 4) or d["min_frac"] < 0.005:
            return None
        return d["n"]
    def gq(kind, c, im, ans, split, pair_id):
        q = f"Does this image contain {sing(c)}?" if kind == "presence" else f"How many {plur(c)} can you see?"
        return dict(image_id=im, image=f"{split}2014/{im}", question=q, target=ans, consensus=vqa_normalize(ans),
                    refs=[ans] * 10, category=c)
    def eval_ground(quota, st, cbi, pool_imgs, tag):
        got = []
        for kind, n in quota.items():
            k = 0
            for c in [catnames[i] for i in rng.permutation(len(catnames))] * 50:
                if k >= n:
                    break
                free = [im for im in pool_imgs if im not in used_val]
                if kind == "presence":
                    P = [im for im in free if pres_pos(st, im, c)]; N = [im for im in free if pres_neg(st, cbi, im, c)]
                    if not P or not N:
                        continue
                    a, b = int(rng.choice(P)), int(rng.choice(N))
                    mem = [gq(kind, c, a, "yes", "val", None), gq(kind, c, b, "no", "val", None)]
                else:
                    C = [(im, count_of(st, im, c)) for im in free]; C = [x for x in C if x[1] is not None]
                    if len({x[1] for x in C}) < 2:
                        continue
                    i = int(rng.integers(len(C))); a, ca = C[i]
                    others = [x for x in C if x[1] != ca]; b, cb = others[int(rng.integers(len(others)))]
                    mem = [gq(kind, c, a, str(ca), "val", None), gq(kind, c, b, str(cb), "val", None)]
                used_val.update([a, b]); k += 1
                q = mem[0]["question"]
                got.append(dict(pair_id=f"gr_{tag}_{kind}_{len(got):03d}", question=q, qtype=kind, split="val",
                                members=mem))
            assert k == n, f"not enough {kind} pairs for {tag}"
        return got
    val_pool = [im for im in ims_va if im in st_va]          # images with instance annotations
    val_pool_sorted = sorted(val_pool)
    sub = sorted(rng.choice(val_pool_sorted, min(6000, len(val_pool_sorted)), replace=False).tolist())
    ground_test = eval_ground(GROUND_TEST, st_va, cbi_va, sub, "test")
    ground_dev = eval_ground(GROUND_DEV, st_va, cbi_va, sub, "dev")
    # dev captions: 96 more unused val images, one caption each
    free_cap = [im for im in ims_va if im not in used_val and len(cbi_va.get(im, [])) >= 1]
    dev_cap_imgs = sorted(rng.choice(free_cap, N_CAP_DEV, replace=False).tolist())
    cap_dev = [dict(image=f"val2014/{im}", caption=cbi_va[im][int(rng.integers(len(cbi_va[im])))]) for im in dev_cap_imgs]
    used_val.update(dev_cap_imgs)
    # training pools (train2014 images that are already used for captions or VQA → no extra downloads)
    train_imgs = sorted(set(cap_imgs) | {m["image_id"] for c in vqa_train for m in c["members"]})
    pres_pool, cnt_pool = [], []
    for c in catnames:
        P = [im for im in train_imgs if pres_pos(st_tr, im, c)]; N = [im for im in train_imgs if pres_neg(st_tr, cbi_tr, im, c)]
        m = min(PRES_PAIRS_PER_CAT, len(P), len(N))
        if m > 0:
            for a, b in zip(rng.choice(P, m, replace=False).tolist(), rng.choice(N, m, replace=False).tolist()):
                mem = [gq("presence", c, a, "yes", "train", None), gq("presence", c, b, "no", "train", None)]
                pres_pool.append(dict(pair_id=f"gr_train_presence_{len(pres_pool):05d}", question=mem[0]["question"],
                                      qtype="presence", split="train", members=mem))
        C = [(im, count_of(st_tr, im, c)) for im in train_imgs]; C = [x for x in C if x[1] is not None]
        if len({x[1] for x in C}) >= 2:
            seen = set()
            for _ in range(COUNT_PAIRS_PER_CAT * 5):
                if len(seen) >= COUNT_PAIRS_PER_CAT:
                    break
                i, j = rng.integers(len(C), size=2)
                (a, ca), (b, cb) = C[i], C[j]
                if ca == cb or (a, b) in seen or (b, a) in seen:
                    continue
                seen.add((a, b))
                mem = [gq("counting", c, a, str(ca), "train", None), gq("counting", c, b, str(cb), "train", None)]
                cnt_pool.append(dict(pair_id=f"gr_train_counting_{len(cnt_pool):05d}", question=mem[0]["question"],
                                     qtype="counting", split="train", members=mem))
    for c in vqa_train + vqa_dev + vqa_test:
        for m in c["members"]:
            m["image"] = f"{c['split']}2014/{m['image_id']}"
    sel.update(caption_train=cap_train, caption_dev=cap_dev, presence_pool=pres_pool, counting_pool=cnt_pool,
               ground_dev=ground_dev, ground_test=ground_test, vqa_train=vqa_train, vqa_dev=vqa_dev, vqa_test=vqa_test)
    # ---------- split-isolation checks ----------
    def imgs(pairs):
        return [m["image"] for c in pairs for m in c["members"]]
    eval_sets = dict(caption_dev=[c["image"] for c in cap_dev], ground_dev=imgs(ground_dev), ground_test=imgs(ground_test),
                     vqa_dev=imgs(vqa_dev), vqa_test=imgs(vqa_test))
    all_eval = [x for v in eval_sets.values() for x in v]
    assert len(all_eval) == len(set(all_eval)), "evaluation images are not unique/disjoint"
    train_set = set([c["image"] for c in cap_train] + imgs(pres_pool) + imgs(cnt_pool) + imgs(vqa_train))
    assert not (train_set & set(all_eval)), "evaluation image leaked into training"
    assert all(x.startswith("val2014/") for x in all_eval) and all(x.startswith("train2014/") for x in train_set)
    sel["summary"] = dict(caption_train=len(cap_train), caption_train_images=len(cap_imgs), caption_dev=len(cap_dev),
                          presence_pool_pairs=len(pres_pool), counting_pool_pairs=len(cnt_pool),
                          ground_dev_pairs=len(ground_dev), ground_test_pairs=len(ground_test),
                          vqa_train_pairs=len(vqa_train), vqa_dev_pairs=len(vqa_dev), vqa_test_pairs=len(vqa_test),
                          train_images=len(train_set), eval_images=len(all_eval),
                          counting_answer_hist=dict(Counter(m["target"] for c in cnt_pool for m in c["members"])),
                          selection_time_s=round(time.time() - t0, 1))
    sel["image_list"] = sorted(train_set) + sorted(all_eval)
    jdump(sel, out)
    print(json.dumps(sel["summary"], indent=1))
    return sel

In [5]:
if RUN_DATA:
    SEL = build_selection()
else:
    SEL = jload(DATA_DIR / "selection.json")
print({k: v for k, v in SEL["summary"].items() if k != "counting_answer_hist"})

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

{
 "caption_train": 18000,
 "caption_train_images": 6000,
 "caption_dev": 96,
 "presence_pool_pairs": 5606,
 "counting_pool_pairs": 5076,
 "ground_dev_pairs": 24,
 "ground_test_pairs": 64,
 "vqa_train_pairs": 1536,
 "vqa_dev_pairs": 32,
 "vqa_test_pairs": 96,
 "train_images": 8755,
 "eval_images": 528,
 "counting_answer_hist": {
  "2": 3840,
  "1": 4726,
  "4": 480,
  "3": 1106
 },
 "selection_time_s": 112.8
}
{'caption_train': 18000, 'caption_train_images': 6000, 'caption_dev': 96, 'presence_pool_pairs': 5606, 'counting_pool_pairs': 5076, 'ground_dev_pairs': 24, 'ground_test_pairs': 64, 'vqa_train_pairs': 1536, 'vqa_dev_pairs': 32, 'vqa_test_pairs': 96, 'train_images': 8755, 'eval_images': 528, 'selection_time_s': 112.8}


## 2 · Image download and SigLIP feature caching (FP16, 196 × 768, after the final LayerNorm)

In [6]:
def fetch_image(key):
    split, iid = key.split("/")
    dest = IMG_DIR / split / f"{iid}.jpg"
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    return download(COCO_IMG_URL.format(split=split, id=int(iid)), dest)

def cache_features(sel):
    meta_p = FEAT_DIR / "features_meta.json"
    if meta_p.exists():
        return jload(meta_p)
    keys = sel["image_list"]
    t0 = time.time()
    with ThreadPoolExecutor(DOWNLOAD_WORKERS) as ex:
        list(ex.map(fetch_image, keys))
    t_dl = time.time() - t0
    from transformers import SiglipVisionModel, AutoImageProcessor
    from PIL import Image
    proc = AutoImageProcessor.from_pretrained(SIGLIP_ID)
    enc = SiglipVisionModel.from_pretrained(SIGLIP_ID, dtype=torch.float32).to(DEVICE).eval()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    D = enc.config.hidden_size
    tmp = SCRATCH / "features.npy"
    arr = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(len(keys), GRID * GRID, D))
    t1 = time.time(); checked = False
    with torch.no_grad():
        for i in range(0, len(keys), 64):
            ims = [Image.open(IMG_DIR / k.split("/")[0] / f"{k.split('/')[1]}.jpg").convert("RGB") for k in keys[i:i + 64]]
            px = proc(images=ims, return_tensors="pt")["pixel_values"].to(DEVICE)
            out = enc(pixel_values=px, output_hidden_states=not checked)
            h = out.last_hidden_state                   # = post_layernorm(encoder output): final LN applied exactly once
            if not checked:
                post_ln = next(m for n, m in enc.named_modules() if n.split(".")[-1] == "post_layernorm")
                ref = post_ln(out.hidden_states[-1])
                assert torch.allclose(ref, h, atol=1e-4), "last_hidden_state is not post-LayerNorm"
                assert h.shape[1] == GRID * GRID, h.shape
                checked = True
            arr[i:i + len(ims)] = h.float().cpu().numpy().astype(np.float16)
    arr.flush(); del arr
    t_enc = time.time() - t1
    peak = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0
    shutil.copy(tmp, FEAT_DIR / "features.npy")
    meta = dict(keys=keys, dim=D, n=len(keys), download_s=round(t_dl, 1), encode_s=round(t_enc, 1),
                peak_gpu_gb=gb(peak), bytes=(FEAT_DIR / "features.npy").stat().st_size,
                preprocessing=dict(processor=proc.to_dict() if hasattr(proc, "to_dict") else str(proc)))
    jdump(meta, meta_p)
    del enc; free_gpu()
    print(f"cached {len(keys)} images: download {t_dl/60:.1f} min, encode {t_enc/60:.1f} min, {gb(meta['bytes'])} GB")
    return meta

def load_features():
    meta = jload(FEAT_DIR / "features_meta.json")
    local = SCRATCH / "features.npy"
    if not local.exists():
        shutil.copy(FEAT_DIR / "features.npy", local)
    arr = np.load(local, mmap_mode="r")
    feats = torch.from_numpy(np.ascontiguousarray(arr))          # fp16, CPU RAM
    return feats, {k: i for i, k in enumerate(meta["keys"])}, meta

In [7]:
if RUN_FEATURES:
    FEAT_META = cache_features(SEL)
    print({k: v for k, v in FEAT_META.items() if k not in ("keys", "preprocessing")})

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  813MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.final_layer_norm.bias      

cached 9283 images: download 5.9 min, encode 3.0 min, 2.603 GB
{'dim': 768, 'n': 9283, 'download_s': 356.8, 'encode_s': 177.5, 'peak_gpu_gb': 1.365, 'bytes': 2794702976}


## 3 · Model: frozen SigLIP features → trainable 2-layer MLP connector → Qwen3-0.6B (frozen FP16, LoRA in stages 2–3)
Visual embeddings replace 196 placeholder positions inserted in the **user** turn *before* the question:
`<|im_start|>user\n[196 visual]\n{question}\n{instruction}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n{answer}<|im_end|>`.
Loss = token CE on answer/caption tokens + `<|im_end|>`, averaged within each example, then over the 16-example batch.

In [8]:
class Connector(nn.Module):
    def __init__(self, d_in, d_lm):
        super().__init__()
        self.fc1, self.act, self.fc2 = nn.Linear(d_in, d_lm), nn.GELU(), nn.Linear(d_lm, d_lm)
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout, gen):
        super().__init__()
        self.base, self.scaling = base, alpha / r
        d_out, d_in = base.weight.shape
        A = torch.empty(r, d_in); A.uniform_(-1 / math.sqrt(d_in), 1 / math.sqrt(d_in), generator=gen)
        self.lora_A = nn.Parameter(A.to(base.weight.device))
        self.lora_B = nn.Parameter(torch.zeros(d_out, r, device=base.weight.device))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        y = self.base(x)
        A, B = self.lora_A, self.lora_B
        if not torch.is_autocast_enabled(x.device.type):
            A, B = A.to(x.dtype), B.to(x.dtype)
        return y + (F.linear(F.linear(self.drop(x), A), B) * self.scaling).to(y.dtype)

def attach_lora(lm):
    gen = torch.Generator().manual_seed(SEED)
    for layer in lm.model.layers:
        for n in LORA_TARGETS:
            setattr(layer.self_attn, n, LoRALinear(getattr(layer.self_attn, n), LORA_R, LORA_ALPHA, LORA_DROPOUT, gen))

def lora_params(lm):
    return {n: p for n, p in lm.named_parameters() if "lora_" in n}

class VLM:
    """Holds tokenizer pieces, the frozen LM, the connector and the feature bank; builds batches for any image condition."""
    def __init__(self, n_vis):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.n_vis = n_vis
        self.tok = AutoTokenizer.from_pretrained(QWEN_ID, revision=QWEN_REV)
        self.lm = AutoModelForCausalLM.from_pretrained(QWEN_ID, revision=QWEN_REV, dtype=torch.float16,
                                                       attn_implementation="sdpa").to(DEVICE)
        self.lm.requires_grad_(False)
        self.lm.config.use_cache = False
        self.lm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        self.feats, self.row, self.feat_meta = load_features()
        self.d_lm = self.lm.config.hidden_size
        self.connector = Connector(self.feat_meta["dim"], self.d_lm).to(DEVICE)    # FP32 trainable
        self.scale = 1.0
        t = self.tok
        self.IMG = t.convert_tokens_to_ids("<|image_pad|>")
        self.EOT = t.convert_tokens_to_ids("<|im_end|>")
        self.PAD = t.pad_token_id if t.pad_token_id is not None else t.convert_tokens_to_ids("<|endoftext|>")
        enc = lambda s: t(s, add_special_tokens=False)["input_ids"]
        self.pre, self.nl = enc("<|im_start|>user\n"), enc("\n")
        self.instr = enc("\n" + ANSWER_INSTR)
        self.post = enc("<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n")
        self.cap_prompt = enc(CAPTION_PROMPT)
        self._enc = enc
        self._check_template()

    def _check_template(self):
        q = "Is this a test?"
        ref = self.tok.apply_chat_template([{"role": "user", "content": "<|image_pad|>\n" + q + "\n" + ANSWER_INSTR}],
                                           tokenize=False, add_generation_prompt=True, enable_thinking=False)
        ids, _ = self.build(q, None, with_image=True, kind="qa")
        mine = self.tok.decode(ids).replace("<|image_pad|>" * self.n_vis, "<|image_pad|>")
        assert mine == ref, f"chat-structure mismatch:\n{mine!r}\n{ref!r}"

    def build(self, question, target, with_image=True, kind="qa", include_eot=True):
        """Token ids + label mask flags. kind: 'qa' (question cap 80, answer cap 16) or 'caption' (cap 64)."""
        if kind == "caption":
            body, cap = self.cap_prompt, CAP_CAPTION
        else:
            body, cap = self._enc(question)[:CAP_Q] + self.instr, CAP_ANS
        vis = ([self.IMG] * self.n_vis + self.nl) if with_image else []
        prompt = self.pre + vis + body + self.post
        if target is None:
            return prompt, None
        tgt = self._enc(target)[:cap]
        lab = tgt + ([self.EOT] if include_eot else [])
        return prompt + tgt + [self.EOT], len(prompt), len(lab)

    def collate(self, items, left_pad=False):
        """items: list of (ids, n_prompt, n_label, feat_row or -1). Returns tensors on DEVICE."""
        T = max(len(x[0]) for x in items)
        B = len(items)
        ids = torch.full((B, T), self.PAD, dtype=torch.long)
        attn = torch.zeros((B, T), dtype=torch.long)
        labels = torch.full((B, T), -100, dtype=torch.long)
        for b, (seq, npr, nlab, _) in enumerate(items):
            off = T - len(seq) if left_pad else 0
            ids[b, off:off + len(seq)] = torch.tensor(seq)
            attn[b, off:off + len(seq)] = 1
            if nlab:
                labels[b, off + npr: off + npr + nlab] = torch.tensor(seq[npr:npr + nlab])
        rows = torch.tensor([x[3] for x in items])
        return ids.to(DEVICE), attn.to(DEVICE), labels.to(DEVICE), rows

    def visual(self, rows):
        F_ = self.feats[rows].to(DEVICE, non_blocking=True).float()          # B × 196 × D
        if self.n_vis == 49:                                                  # 2×2 average pooling of the 14×14 grid
            B, _, D = F_.shape
            F_ = F_.view(B, 7, 2, 7, 2, D).mean(dim=(2, 4)).reshape(B, 49, D)
        return self.connector(F_) * self.scale

    def embed(self, ids, rows):
        emb = self.lm.get_input_embeddings()(ids)
        has = rows >= 0
        if has.any():
            vis = self.visual(rows[has])
            mask = (ids == self.IMG)
            assert int(mask.sum()) == int(has.sum()) * self.n_vis
            emb = emb.masked_scatter(mask.unsqueeze(-1), vis.to(emb.dtype))
        return emb

    def per_example_nll(self, ids, attn, labels, rows):
        """Mean token NLL of the labelled tokens of each example (B,)."""
        emb = self.embed(ids, rows)
        pos = (attn.cumsum(-1) - 1).clamp(min=0)
        h = self.lm.model(inputs_embeds=emb, attention_mask=attn, position_ids=pos, use_cache=False).last_hidden_state
        tgt = labels[:, 1:]; m = tgt != -100
        logits = self.lm.lm_head(h[:, :-1][m]).float()
        nll = F.cross_entropy(logits, tgt[m], reduction="none")
        ex = m.nonzero()[:, 0]
        s = torch.zeros(ids.shape[0], device=ids.device).index_add_(0, ex, nll)
        c = torch.zeros(ids.shape[0], device=ids.device).index_add_(0, ex, torch.ones_like(nll))
        return s / c.clamp(min=1)

    @torch.no_grad()
    def greedy(self, ids, attn, rows, max_new=MAX_NEW_TOKENS):
        emb = self.embed(ids, rows)
        pos = (attn.cumsum(-1) - 1).clamp(min=0)
        out = self.lm.model(inputs_embeds=emb, attention_mask=attn, position_ids=pos, use_cache=True)
        past, h = out.past_key_values, out.last_hidden_state[:, -1]
        B = ids.shape[0]
        done = torch.zeros(B, dtype=torch.bool, device=ids.device)
        gen = []
        for t in range(max_new):
            nxt = self.lm.lm_head(h).float().argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, self.PAD), nxt)
            gen.append(nxt)
            done |= (nxt == self.EOT) | (nxt == self.PAD)
            if done.all() or t == max_new - 1:
                break
            attn = torch.cat([attn, torch.ones((B, 1), dtype=attn.dtype, device=attn.device)], 1)
            pos = pos[:, -1:] + 1
            out = self.lm.model(input_ids=nxt[:, None], attention_mask=attn, position_ids=pos,
                                past_key_values=past, use_cache=True)
            past, h = out.past_key_values, out.last_hidden_state[:, -1]
        G = torch.stack(gen, 1).tolist()
        texts = []
        for g in G:
            cut = [k for k, x in enumerate(g) if x in (self.EOT, self.PAD)]
            texts.append(self.tok.decode(g[: cut[0]] if cut else g, skip_special_tokens=True).strip())
        return texts

    def trainable_state(self):
        return dict(connector=self.connector.state_dict(), lora={n: p.detach().clone() for n, p in lora_params(self.lm).items()},
                    scale=self.scale)

def count_params(module_or_dict):
    if isinstance(module_or_dict, dict):
        return int(sum(p.numel() for p in module_or_dict.values()))
    return int(sum(p.numel() for p in module_or_dict.parameters()))

## 4 · Examples, sampling schedules, connector-scale calibration, dev evaluation

In [9]:
def ex_caption(vlm, c, with_image=True):
    ids, npr, nlab = vlm.build(None, c["caption"], with_image, kind="caption")
    return (ids, npr, nlab, vlm.row[c["image"]] if with_image else -1)

def ex_qa(vlm, question, target, image_key, include_eot=True):
    ids, npr, nlab = vlm.build(question, target, image_key is not None, kind="qa", include_eot=include_eot)
    return (ids, npr, nlab, vlm.row[image_key] if image_key is not None else -1)

def stage_schedule(stage_idx, sel, run_dir):
    """Deterministic list of draws for a stage (each draw = two examples: a complete pair or two replay captions)."""
    p = run_dir / f"schedule_stage{stage_idx+1}.json"
    if p.exists():
        return jload(p)
    st = STAGES[stage_idx]
    rng = np.random.default_rng([SEED, stage_idx + 1])
    n_draws = st["updates"] * EFF_BS // 2
    nc = len(sel["caption_train"])
    if stage_idx == 0:
        order = np.concatenate([rng.permutation(nc), rng.permutation(nc)]).tolist()       # two passes
        draws = [("caption", [order[2 * i], order[2 * i + 1]]) for i in range(n_draws)]
    else:
        kinds, probs = list(st["mix"]), np.array(list(st["mix"].values()))
        pools = dict(presence=len(sel["presence_pool"]), counting=len(sel["counting_pool"]), vqa=len(sel["vqa_train"]))
        draws = []
        for _ in range(n_draws):
            k = kinds[int(rng.choice(len(kinds), p=probs))]
            draws.append((k, rng.integers(nc, size=2).tolist()) if k == "caption" else (k, [int(rng.integers(pools[k]))]))
    sched = dict(draws=draws, mixture_counts=dict(Counter(k for k, _ in draws)))
    jdump(sched, p)
    return sched

def draw_examples(vlm, sel, draw):
    k, idx = draw
    if k == "caption":
        return [ex_caption(vlm, sel["caption_train"][i]) for i in idx]
    pool = dict(presence=sel["presence_pool"], counting=sel["counting_pool"], vqa=sel["vqa_train"])[k]
    pair = pool[idx[0]]
    return [ex_qa(vlm, pair["question"], m["target"], m["image"]) for m in pair["members"]]

@torch.no_grad()
def calibrate_scale(vlm, sel):
    """Rule: scale = mean ‖e‖₂ of Qwen input embeddings over training-text tokens / mean ‖g_φ0(F)‖₂ over
    training-image visual tokens (initial connector). Training data only; computed once and frozen."""
    rng = np.random.default_rng(SEED)
    texts = [c["caption"] for c in sel["caption_train"][:3000]] + [c["question"] for c in sel["vqa_train"]] + \
            [c["question"] for c in sel["presence_pool"][:500] + sel["counting_pool"][:500]]
    ids = torch.tensor([i for t in texts for i in vlm._enc(t)], device=DEVICE)
    text_norm = float(vlm.lm.get_input_embeddings()(ids).float().norm(dim=-1).mean())
    keys = [c["image"] for c in sel["caption_train"]]
    pick = rng.choice(len(keys), min(256, len(keys)), replace=False)
    rows = torch.tensor([vlm.row[keys[i]] for i in pick])
    vlm.scale = 1.0
    with autocast():
        v = torch.cat([vlm.visual(rows[i:i + 32]).float() for i in range(0, len(rows), 32)])
    conn_norm = float(v.norm(dim=-1).mean())
    return dict(rule="scale = mean||text_emb|| (training captions+questions) / mean||connector_init(F)|| (256 training images)",
                text_norm=text_norm, connector_init_norm=conn_norm, scale=text_norm / conn_norm)

def dev_suites(sel):
    return dict(ground=sel["ground_dev"], vqa=sel["vqa_dev"])

@torch.no_grad()
def batched_nll(vlm, items, bs=16):
    out = []
    for i in range(0, len(items), bs):
        ids, attn, lab, rows = vlm.collate(items[i:i + bs])
        with autocast():
            out.append(vlm.per_example_nll(ids, attn, lab, rows).float().cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def dev_evaluate(vlm, sel):
    vlm.lm.eval(); vlm.connector.eval()
    res = {}
    caps = sel["caption_dev"]
    res["caption_loss"] = float(batched_nll(vlm, [ex_caption(vlm, c) for c in caps]).mean())
    res["caption_loss_noimage"] = float(batched_nll(vlm, [ex_caption(vlm, c, with_image=False) for c in caps]).mean())
    for suite, pairs in dev_suites(sel).items():
        for cond in ("correct", "complementary", "noimage"):
            items, types = [], []
            for p in pairs:
                for j, m in enumerate(p["members"]):
                    img = {"correct": m["image"], "complementary": p["members"][1 - j]["image"], "noimage": None}[cond]
                    items.append(ex_qa(vlm, p["question"], m["target"], img, include_eot=False)); types.append(p["qtype"])
            nll = batched_nll(vlm, items)
            res[f"{suite}_nll_{cond}"] = float(nll.mean())
            for t in sorted(set(types)):
                res[f"{suite}_nll_{cond}_{t.replace('/', '')}"] = float(nll[np.array(types) == t].mean())
            res.setdefault(f"_{suite}", {})[cond] = nll
        res[f"{suite}_swap_penalty"] = float((res[f"_{suite}"]["complementary"] - res[f"_{suite}"]["correct"]).mean())
        res.pop(f"_{suite}")
    vlm.lm.train(); vlm.connector.train()
    return res

## 5 · Training (three stages, resumable every 250 updates)

In [10]:
def stage_of(u):
    """Global update index u (0-based) → (stage index, step within stage)."""
    start = 0
    for i, s in enumerate(STAGES):
        if u < start + s["updates"]:
            return i, u - start
        start += s["updates"]
    return len(STAGES) - 1, STAGES[-1]["updates"]

def stage_lambda(n):
    w = max(1, math.ceil(WARMUP_FRAC * n))
    def f(s):
        if s < w:
            return (s + 1) / w
        p = min(1.0, (s - w) / max(1, n - w))
        return MIN_LR_FRAC + (1 - MIN_LR_FRAC) * 0.5 * (1 + math.cos(math.pi * p))
    return f

def make_optimizer(vlm, stage_idx):
    st = STAGES[stage_idx]
    cd = [p for n, p in vlm.connector.named_parameters() if p.ndim >= 2]
    cn = [p for n, p in vlm.connector.named_parameters() if p.ndim < 2]
    groups = [dict(params=cd, lr=st["conn_lr"], weight_decay=WEIGHT_DECAY), dict(params=cn, lr=st["conn_lr"], weight_decay=0.0)]
    if st["lora_lr"] is not None:
        groups.append(dict(params=list(lora_params(vlm.lm).values()), lr=st["lora_lr"], weight_decay=WEIGHT_DECAY))
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, stage_lambda(st["updates"]))
    return opt, sched

def rng_state():
    return dict(py=random.getstate(), np=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

def set_rng_state(s):
    random.setstate(s["py"]); np.random.set_state(s["np"]); torch.set_rng_state(s["torch"])
    if s["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(s["cuda"])

def train_run(run_name, n_vis, sel):
    run_dir = RUNS_DIR / run_name; run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "TRAIN_DONE").exists():
        print(f"[{run_name}] training already finished"); return
    set_seed(SEED)
    vlm = VLM(n_vis)
    ckdir = run_dir / "checkpoints"; ckdir.mkdir(exist_ok=True)
    cks = sorted(ckdir.glob("ckpt_u*.pt"))
    state = dict(u=0, stage_time_s=[0.0] * len(STAGES), peak_alloc=[0] * len(STAGES), trainable=[0] * len(STAGES),
                 skipped=[], mixture=[{} for _ in STAGES])
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    if cks:
        ck = torch.load(cks[-1], map_location="cpu", weights_only=False)
        state = ck["state"]
        vlm.connector.load_state_dict(ck["connector"]); vlm.scale = ck["scale"]
        if ck["lora"]:
            attach_lora(vlm.lm)
            live = lora_params(vlm.lm)
            with torch.no_grad():
                for n, t in ck["lora"].items():
                    live[n].copy_(t)
        si, step = stage_of(state["u"])
        if step == 0 or ck["opt"] is None:      # checkpoint sits on a stage boundary → new stage starts in the loop
            opt = sched = None; cur_stage = si - 1
        else:
            opt, sched = make_optimizer(vlm, si); cur_stage = si
            opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); set_rng_state(ck["rng"])
        print(f"[{run_name}] resumed at update {state['u']} (stage {si+1}, step {step})")
        logs = [r for r in read_jsonl(run_dir / "train_log.jsonl") if r["update"] <= state["u"]]
        with open(run_dir / "train_log.jsonl", "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in logs)
        dl = [r for r in read_jsonl(run_dir / "dev_log.jsonl") if r["update"] <= state["u"]]
        with open(run_dir / "dev_log.jsonl", "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in dl)
        del ck
    else:
        for f in ("train_log.jsonl", "dev_log.jsonl"):
            (run_dir / f).unlink(missing_ok=True)
        cal = calibrate_scale(vlm, sel); vlm.scale = cal["scale"]
        jdump(cal, run_dir / "calibration.json"); print("calibration:", cal)
        opt, sched = make_optimizer(vlm, 0); cur_stage = 0
    # one-off interface checks
    vlm.connector.train(); vlm.lm.train()
    frozen_ok = all(not p.requires_grad for n, p in vlm.lm.named_parameters() if "lora_" not in n)
    assert frozen_ok, "an original Qwen parameter is trainable"
    scheds = [stage_schedule(i, sel, run_dir) for i in range(len(STAGES))]
    dev_done = {r["update"] for r in read_jsonl(run_dir / "dev_log.jsonl")}
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t_seg = time.time(); t_print = time.time()
    def run_dev(u):
        if u in dev_done or u not in DEV_EVAL_AT:
            return
        t = time.time(); r = dev_evaluate(vlm, sel); r.update(update=u, eval_s=round(time.time() - t, 1))
        append_jsonl(run_dir / "dev_log.jsonl", r); dev_done.add(u)
        print(f"  [dev u={u}] cap={r['caption_loss']:.3f} | ground nll c/s/n={r['ground_nll_correct']:.3f}/"
              f"{r['ground_nll_complementary']:.3f}/{r['ground_nll_noimage']:.3f} | vqa nll c/s/n={r['vqa_nll_correct']:.3f}/"
              f"{r['vqa_nll_complementary']:.3f}/{r['vqa_nll_noimage']:.3f}")
    t0 = time.time(); run_dev(state["u"]); t_seg += time.time() - t0
    first_grad_check = state["u"] == 0
    while state["u"] < TOTAL_UPDATES:
        u = state["u"]
        si, step = stage_of(u)
        if si != cur_stage:
            # ---- stage transition: enable LoRA (stage 2), fresh optimizer + per-stage warm-up/cosine schedule ----
            if STAGES[si]["lora_lr"] is not None and not lora_params(vlm.lm):
                attach_lora(vlm.lm); vlm.lm.train()
            opt, sched = make_optimizer(vlm, si)
            cur_stage = si
            print(f"  → stage {si+1} ({STAGES[si]['name']}): trainable params = "
                  f"{count_params(vlm.connector) + count_params(lora_params(vlm.lm)):,}")
        state["trainable"][si] = count_params(vlm.connector) + count_params(lora_params(vlm.lm))
        draws = scheds[si]["draws"][step * EFF_BS // 2:(step + 1) * EFF_BS // 2]
        for k, _ in draws:
            state["mixture"][si][k] = state["mixture"][si].get(k, 0) + 1
        loss_acc = 0.0
        for mb in range(GRAD_ACC):
            items = [e for d in draws[mb * MICRO_BS // 2:(mb + 1) * MICRO_BS // 2] for e in draw_examples(vlm, sel, d)]
            ids, attn, lab, rows = vlm.collate(items)
            with autocast():
                per_ex = vlm.per_example_nll(ids, attn, lab, rows)
                loss = per_ex.sum() / EFF_BS
            scaler.scale(loss).backward()
            loss_acc += float(loss.detach())
        if first_grad_check:
            g = vlm.connector.fc1.weight.grad
            assert g is not None, "no gradient reached the connector"
            finite = bool(torch.isfinite(g).all())
            print(f"  first-step connector grad: nonzero={bool((g != 0).any())} finite={finite} "
                f"loss={loss_acc:.4f} loss_scale={scaler.get_scale() if scaler.is_enabled() else 1.0:.0f}"
                + ("" if finite else " (FP16 overflow at the initial loss scale: GradScaler skips this "
                                          "update and lowers the scale; expected on the first update(s))"))
            assert bool((g != 0).any()), "connector gradient is identically zero"
            first_grad_check = False
        scaler.unscale_(opt)
        params = [p for g in opt.param_groups for p in g["params"]]
        gn = float(torch.nn.utils.clip_grad_norm_(params, CLIP))
        sb = scaler.get_scale() if scaler.is_enabled() else 1.0
        lrs = sched.get_last_lr()
        scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
        skipped = scaler.is_enabled() and scaler.get_scale() < sb
        if skipped:
            state["skipped"].append(u + 1)
        state["u"] = u + 1
        append_jsonl(run_dir / "train_log.jsonl", dict(update=u + 1, stage=si + 1, loss=loss_acc, grad_norm=gn,
                                                       lr_connector=lrs[0], lr_lora=lrs[2] if len(lrs) > 2 else None,
                                                       skipped=bool(skipped)))
        if (u + 1) % 50 == 0:
            spu = (time.time() - t_print) / 50; t_print = time.time()
            print(f"  u={u+1} stage={si+1} loss={loss_acc:.4f} |g|={gn:.2f} lr={lrs[0]:.2e} "
                  f"{spu:.2f}s/update, ETA {(TOTAL_UPDATES-u-1)*spu/60:.0f} min")
        end_of_stage = (u + 1) in STAGE_ENDS
        need_ck = (u + 1) % CKPT_EVERY == 0 or end_of_stage
        if (u + 1) in DEV_EVAL_AT or need_ck:
            if torch.cuda.is_available():
                state["peak_alloc"][si] = max(state["peak_alloc"][si], torch.cuda.max_memory_allocated())
                torch.cuda.reset_peak_memory_stats()
            state["stage_time_s"][si] += time.time() - t_seg
            if end_of_stage:
                torch.save(vlm.trainable_state(), run_dir / f"weights_after_stage{si+1}.pt")
            run_dev(u + 1)
            if need_ck:
                torch.save(dict(state=state, connector=vlm.connector.state_dict(), scale=vlm.scale,
                                lora={n: p.detach().cpu() for n, p in lora_params(vlm.lm).items()},
                                opt=None if end_of_stage else opt.state_dict(),
                                sched=None if end_of_stage else sched.state_dict(),
                                scaler=scaler.state_dict(), rng=rng_state(),
                                sampler_position=dict(stage=si + 1, step_in_stage=step + 1)),
                           ckdir / f"ckpt_u{u+1:04d}.pt")
            t_seg = time.time()
    summary = dict(run=run_name, n_vis=n_vis, stage_time_s=state["stage_time_s"],
                   stage_peak_alloc_gb=[gb(x) for x in state["peak_alloc"]], trainable_params=state["trainable"],
                   connector_params=count_params(vlm.connector), lora_params=count_params(lora_params(vlm.lm)),
                   mixture_counts=state["mixture"], skipped_updates=state["skipped"], scale=vlm.scale,
                   gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
    jdump(summary, run_dir / "train_summary.json")
    (run_dir / "TRAIN_DONE").touch()
    del vlm; free_gpu()
    print(f"[{run_name}] training done: {sum(state['stage_time_s'])/3600:.2f} h")

In [11]:
RUN_LIST = [("vis196", 196)] + ([("vis49", 49)] if RUN_COMPRESSED else [])
if RUN_TRAIN:
    for _name, _nv in RUN_LIST:
        train_run(_name, _nv, SEL)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/tmp/ipykernel_23/2369882182.py:59: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  feats = torch.from_numpy(np.ascontiguousarray(arr))          # fp16, CPU RAM


calibration: {'rule': 'scale = mean||text_emb|| (training captions+questions) / mean||connector_init(F)|| (256 training images)', 'text_norm': 0.8681159019470215, 'connector_init_norm': 15.857780456542969, 'scale': 0.05474384667677967}
  [dev u=0] cap=5.401 | ground nll c/s/n=9.326/9.045/5.274 | vqa nll c/s/n=9.818/9.802/4.867
  first-step connector grad: nonzero=True finite=False loss=5.2106 loss_scale=65536 (FP16 overflow at the initial loss scale: GradScaler skips this update and lowers the scale; expected on the first update(s))


/tmp/ipykernel_23/1552120287.py:139: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


  u=50 stage=1 loss=3.5527 |g|=6.01 lr=4.42e-04 1.44s/update, ETA 107 min
  u=100 stage=1 loss=3.0221 |g|=2.28 lr=8.85e-04 1.24s/update, ETA 91 min
  u=150 stage=1 loss=3.2374 |g|=1.19 lr=9.99e-04 1.23s/update, ETA 89 min
  u=200 stage=1 loss=3.0930 |g|=0.98 lr=9.96e-04 1.23s/update, ETA 88 min
  u=250 stage=1 loss=2.9199 |g|=2.62 lr=9.91e-04 1.23s/update, ETA 87 min
  u=300 stage=1 loss=2.6243 |g|=0.83 lr=9.83e-04 1.25s/update, ETA 87 min
  u=350 stage=1 loss=2.9123 |g|=0.82 lr=9.73e-04 1.24s/update, ETA 86 min
  u=400 stage=1 loss=2.5991 |g|=0.73 lr=9.61e-04 1.24s/update, ETA 85 min
  u=450 stage=1 loss=2.4295 |g|=0.88 lr=9.46e-04 1.24s/update, ETA 84 min
  u=500 stage=1 loss=2.5921 |g|=0.82 lr=9.29e-04 1.24s/update, ETA 83 min
  [dev u=500] cap=2.536 | ground nll c/s/n=3.416/3.621/5.274 | vqa nll c/s/n=4.103/4.269/4.867
  u=550 stage=1 loss=2.5877 |g|=0.85 lr=9.11e-04 1.39s/update, ETA 91 min
  u=600 stage=1 loss=2.5155 |g|=0.72 lr=8.90e-04 1.24s/update, ETA 80 min
  u=650 stage=1 l

## 6 · Final test evaluation: correct / complementary swap / mismatched / no image
Greedy, thinking disabled, 16 new tokens, batch 8. References are always the original question's references. Grounding: normalised exact match. Natural VQAv2: VQA soft accuracy against all ten human answers.

In [12]:
CONDITIONS = ["correct", "complementary", "mismatched", "noimage"]

def mismatch_map(pairs, tag):
    p = DATA_DIR / f"mismatch_map_{tag}.json"
    if p.exists():
        return jload(p)
    rng = np.random.default_rng([SEED, len(pairs)])
    while True:
        perm = rng.permutation(len(pairs))
        if all(perm[i] != i for i in range(len(pairs))):
            break
    mp = {pairs[i]["pair_id"]: pairs[int(perm[i])]["pair_id"] for i in range(len(pairs))}
    jdump(mp, p)
    return mp

def load_trained(run_name, n_vis):
    vlm = VLM(n_vis)
    w = torch.load(RUNS_DIR / run_name / f"weights_after_stage{len(STAGES)}.pt", map_location="cpu", weights_only=False)
    vlm.connector.load_state_dict(w["connector"]); vlm.scale = w["scale"]
    attach_lora(vlm.lm)
    live = lora_params(vlm.lm)
    with torch.no_grad():
        for n, t in w["lora"].items():
            live[n].copy_(t)
    vlm.lm.eval(); vlm.connector.eval()
    return vlm

@torch.no_grad()
def final_evaluate(run_name, n_vis, sel):
    out_p = RUNS_DIR / run_name / "final_test_records.json"
    if out_p.exists():
        return jload(out_p)
    vlm = load_trained(run_name, n_vis)
    recs = []
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for suite, pairs in (("ground", sel["ground_test"]), ("vqa", sel["vqa_test"])):
        mm = mismatch_map(pairs, suite)
        byid = {p["pair_id"]: p for p in pairs}
        for cond in CONDITIONS:
            qs = []
            for p in pairs:
                for j, m in enumerate(p["members"]):
                    img = {"correct": m["image"], "complementary": p["members"][1 - j]["image"],
                           "mismatched": byid[mm[p["pair_id"]]]["members"][j]["image"], "noimage": None}[cond]
                    qs.append((p, j, m, img))
            nll = batched_nll(vlm, [ex_qa(vlm, p["question"], m["target"], img, include_eot=False) for p, j, m, img in qs])
            preds, gen_s = [], 0.0
            for i in range(0, len(qs), GEN_BATCH):
                chunk = qs[i:i + GEN_BATCH]
                items = []
                for p, j, m, img in chunk:
                    ids = vlm.build(p["question"], None, img is not None, kind="qa")[0]
                    items.append((ids, len(ids), 0, vlm.row[img] if img is not None else -1))
                ids, attn, _, rows = vlm.collate(items, left_pad=True)
                cuda_sync(); t = time.time()
                with autocast():
                    preds += vlm.greedy(ids, attn, rows)
                cuda_sync(); gen_s += time.time() - t
            for (p, j, m, img), pred, l in zip(qs, preds, nll):
                pn = vqa_normalize(pred)
                refs_n = [vqa_normalize(r) for r in m["refs"]]
                partner = p["members"][1 - j]
                acc = vqa_soft_acc(pn, refs_n) if suite == "vqa" else float(pn == m["consensus"])
                recs.append(dict(suite=suite, condition=cond, pair_id=p["pair_id"], member=j, qtype=p["qtype"],
                                 question=p["question"], image_used=img, image_original=m["image"],
                                 target=m["target"], consensus=m["consensus"], partner_consensus=partner["consensus"],
                                 refs=m["refs"], pred=pred, pred_norm=pn, acc=acc,
                                 exact_canonical=float(pn == m["consensus"]),
                                 follows_partner=float(pn == partner["consensus"]), empty=float(pn == ""),
                                 ref_nll=float(l), gen_s_per_answer=gen_s / len(qs)))
            print(f"  {suite:6s} {cond:13s} acc={np.mean([r['acc'] for r in recs if r['suite']==suite and r['condition']==cond]):.3f}")
    info = dict(peak_gpu_gb=gb(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0)
    jdump(info, RUNS_DIR / run_name / "final_eval_info.json")
    jdump(recs, out_p)
    del vlm; free_gpu()
    return recs

def pair_table(recs):
    import pandas as pd
    df = pd.DataFrame(recs)
    g = df.groupby(["suite", "condition", "pair_id"])
    pt = g.agg(qtype=("qtype", "first"), acc=("acc", "mean"), both_canonical=("exact_canonical", "min"),
               nll=("ref_nll", "mean")).reset_index()
    ident = g.apply(lambda x: float(x.sort_values("member")["pred_norm"].nunique() == 1), include_groups=False).rename("identical").reset_index()
    return pt.merge(ident, on=["suite", "condition", "pair_id"])

def boot_ci(values_by_pair, n=N_BOOT, seed=SEED):
    """values_by_pair: array (n_pairs,) of per-pair statistics → mean and 95% CI by resampling complete pairs."""
    v = np.asarray(values_by_pair, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(v), size=(n, len(v)))
    bs = v[idx].mean(1)
    return float(v.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

def summarize_final(run_name, recs):
    import pandas as pd
    df = pd.DataFrame(recs)
    pt = pair_table(recs)
    rows = []
    for (suite, cond), d in df.groupby(["suite", "condition"]):
        p = pt[(pt.suite == suite) & (pt.condition == cond)].sort_values("pair_id")
        m, lo, hi = boot_ci(p["acc"].values)
        pm, plo, phi = boot_ci(p["both_canonical"].values)
        row = dict(run=run_name, suite=suite, condition=cond, n_questions=len(d), accuracy=m, acc_ci_lo=lo, acc_ci_hi=hi,
                   pair_accuracy=pm, pair_acc_ci_lo=plo, pair_acc_ci_hi=phi, identical_answer_rate=p["identical"].mean(),
                   empty_answer_rate=d["empty"].mean(), mean_ref_nll=d["ref_nll"].mean(),
                   follows_partner_rate=d["follows_partner"].mean(), gen_s_per_answer=d["gen_s_per_answer"].iloc[0])
        for t, dt in d.groupby("qtype"):
            row[f"acc_{t}"] = dt["acc"].mean()
        rows.append(row)
    main = pd.DataFrame(rows)
    adv = []
    for suite in df.suite.unique():
        base = pt[(pt.suite == suite) & (pt.condition == "correct")].set_index("pair_id").sort_index()
        for cond in ("complementary", "mismatched", "noimage"):
            other = pt[(pt.suite == suite) & (pt.condition == cond)].set_index("pair_id").loc[base.index]
            da = boot_ci(base["acc"].values - other["acc"].values)
            dn = boot_ci(other["nll"].values - base["nll"].values)
            dp = boot_ci(base["both_canonical"].values - other["both_canonical"].values)
            adv.append(dict(run=run_name, suite=suite, vs=cond, acc_advantage=da[0], acc_adv_ci=(round(da[1], 4), round(da[2], 4)),
                            nll_advantage=dn[0], nll_adv_ci=(round(dn[1], 4), round(dn[2], 4)),
                            pair_acc_advantage=dp[0], pair_acc_adv_ci=(round(dp[1], 4), round(dp[2], 4))))
    adv = pd.DataFrame(adv)
    main.to_csv(TAB_DIR / f"final_{run_name}.csv", index=False)
    adv.to_csv(TAB_DIR / f"final_advantages_{run_name}.csv", index=False)
    df.drop(columns=["refs"]).to_csv(TAB_DIR / f"final_per_example_{run_name}.csv", index=False)
    return main, adv

In [13]:
if RUN_FINAL_EVAL:
    for _name, _nv in RUN_LIST:
        if not (RUNS_DIR / _name / "TRAIN_DONE").exists():
            print(f"[{_name}] not trained yet"); continue
        _recs = final_evaluate(_name, _nv, SEL)
        _main, _adv = summarize_final(_name, _recs)
        print(_main.round(3).to_string(index=False)); print(_adv.round(3).to_string(index=False))

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  ground correct       acc=0.797
  ground complementary acc=0.148
  ground mismatched    acc=0.422
  ground noimage       acc=0.422
  vqa    correct       acc=0.668
  vqa    complementary acc=0.225
  vqa    mismatched    acc=0.366
  vqa    noimage       acc=0.330
   run  suite     condition  n_questions  accuracy  acc_ci_lo  acc_ci_hi  pair_accuracy  pair_acc_ci_lo  pair_acc_ci_hi  identical_answer_rate  empty_answer_rate  mean_ref_nll  follows_partner_rate  gen_s_per_answer  acc_counting  acc_presence  acc_number  acc_other  acc_yes/no
vis196 ground complementary          128     0.148      0.086      0.211          0.031           0.000           0.078                  0.234                0.0         3.763                 0.797             0.034         0.250         0.047         NaN        NaN         NaN
vis196 ground       correct          128     0.797      0.727      0.859          0.625           0.500           0.734                  0.234                0.0         0.645   

## 7 · Analysis: learning curves, qualitative aligned examples, resources

In [14]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def plot_curves(run_name):
    tl = pd.DataFrame(read_jsonl(RUNS_DIR / run_name / "train_log.jsonl"))
    dl = pd.DataFrame(read_jsonl(RUNS_DIR / run_name / "dev_log.jsonl")).sort_values("update").drop_duplicates("update", keep="last")
    dl.to_csv(TAB_DIR / f"dev_curves_{run_name}.csv", index=False)
    fig, ax = plt.subplots(1, 4, figsize=(19, 3.8))
    ax[0].plot(tl["update"], tl["loss"], alpha=0.25, lw=0.6); ax[0].plot(tl["update"], tl["loss"].rolling(50, min_periods=1).mean(), lw=1.2)
    ax[0].set_title("training loss (example-weighted)"); ax[1].plot(dl["update"], dl["caption_loss"], "o-", label="correct image")
    ax[1].plot(dl["update"], dl["caption_loss_noimage"], "x--", label="no image"); ax[1].set_title("dev caption reference loss"); ax[1].legend(fontsize=8)
    for suite, a in (("ground", ax[2]), ("vqa", ax[2])):
        for cond, ls in (("correct", "-"), ("complementary", "--"), ("noimage", ":")):
            a.plot(dl["update"], dl[f"{suite}_nll_{cond}"], ls, marker="o" if suite == "ground" else "s", ms=3,
                   color="C0" if suite == "ground" else "C1", label=f"{suite} · {cond}")
    ax[2].set_title("dev answer NLL (excl. end-of-turn)"); ax[2].legend(fontsize=7)
    ax[3].plot(dl["update"], dl["ground_swap_penalty"], "o-", label="annotation-derived")
    ax[3].plot(dl["update"], dl["vqa_swap_penalty"], "s-", label="natural VQAv2"); ax[3].axhline(0, color="k", lw=0.6)
    ax[3].set_title("image-swap penalty ΔNLL"); ax[3].legend(fontsize=8)
    for a in ax:
        for e in STAGE_ENDS[:-1]:
            a.axvline(e, color="grey", lw=0.8, ls="--")
        a.set_xlabel("main-run update")
    fig.tight_layout(); fig.savefig(FIG_DIR / f"learning_curves_{run_name}.png", dpi=200); plt.close(fig)

def qualitative(run_name, recs, sel):
    """Selection rule (fixed before looking at outputs): within each suite and question type, take the first pair in
    pair-id order for each of three behaviours — (a) both answers correct with correct images, (b) the swapped-image
    prediction follows the partner's answer, (c) identical predictions for both images under correct inputs."""
    df = pd.DataFrame(recs)
    pt = pair_table(recs)
    chosen = []
    for (suite, qt), g in pt[pt.condition == "correct"].groupby(["suite", "qtype"]):
        g = g.sort_values("pair_id")
        sw = df[(df.suite == suite) & (df.condition == "complementary")].groupby("pair_id")["follows_partner"].min()
        rules = {"both_correct": g[g.both_canonical == 1], "swap_follows_partner": g[g.pair_id.isin(sw[sw == 1].index)],
                 "identical_answers": g[g.identical == 1]}
        for rule, cand in rules.items():
            if len(cand):
                chosen.append((suite, qt, rule, cand.iloc[0]["pair_id"]))
    lines = []
    for suite, qt, rule, pid in chosen:
        d = df[df.pair_id == pid]
        q = d.iloc[0]["question"]
        lines.append(f"### {suite} · {qt} · rule={rule} · {pid}\n**Q:** {q}\n")
        lines.append("| image | references (canonical) | correct | complementary | mismatched | no image |\n|---|---|---|---|---|---|")
        for j in (0, 1):
            dj = d[d.member == j].set_index("condition")
            lines.append(f"| {dj.loc['correct','image_original']} | {dj.loc['correct','target']} ({dj.loc['correct','consensus']}) | "
                         + " | ".join(f"{dj.loc[c,'pred']!s} ({dj.loc[c,'acc']:.2f})" for c in CONDITIONS) + " |")
        lines.append("")
    (TAB_DIR / f"qualitative_{run_name}.md").write_text("\n".join(lines))
    # figure with the complementary image pairs (if the images are still on local disk, they are re-downloaded otherwise)
    try:
        from PIL import Image
        sel_pairs = chosen[:6]
        fig, axes = plt.subplots(len(sel_pairs), 2, figsize=(8, 3.2 * len(sel_pairs)))
        axes = np.atleast_2d(axes)
        for r, (suite, qt, rule, pid) in enumerate(sel_pairs):
            d = df[(df.pair_id == pid) & (df.condition == "correct")].sort_values("member")
            for j, (_, x) in enumerate(d.iterrows()):
                img = Image.open(fetch_image(x["image_original"])).convert("RGB")
                axes[r, j].imshow(img); axes[r, j].axis("off")
                sw = df[(df.pair_id == pid) & (df.condition == "complementary") & (df.member == j)].iloc[0]["pred"]
                axes[r, j].set_title(f"{x['question'][:45]}\nref={x['target']} | pred={x['pred']} | swap-pred={sw}", fontsize=7)
        fig.tight_layout(); fig.savefig(FIG_DIR / f"qualitative_pairs_{run_name}.png", dpi=150); plt.close(fig)
    except Exception as e:
        print("qualitative figure skipped:", e)
    return chosen

def resources_table():
    rows = []
    fm = jload(FEAT_DIR / "features_meta.json") if (FEAT_DIR / "features_meta.json").exists() else {}
    if fm:
        rows.append(dict(item="feature caching (download + SigLIP)", time_min=round((fm["download_s"] + fm["encode_s"]) / 60, 1),
                         peak_gpu_gb=fm["peak_gpu_gb"], trainable_params=0, notes=f"{fm['n']} images, {gb(fm['bytes'])} GB FP16 cache"))
    for name, _ in [("vis196", 196), ("vis49", 49)]:
        f = RUNS_DIR / name / "train_summary.json"
        if not f.exists():
            continue
        s = jload(f)
        for i, st in enumerate(STAGES):
            rows.append(dict(item=f"{name} · {st['name']}", time_min=round(s["stage_time_s"][i] / 60, 1),
                             peak_gpu_gb=s["stage_peak_alloc_gb"][i], trainable_params=s["trainable_params"][i],
                             notes=f"mixture draws {s['mixture_counts'][i]}"))
        rf = RUNS_DIR / name / "final_test_records.json"
        if rf.exists():
            r = pd.DataFrame(jload(rf))
            info = jload(RUNS_DIR / name / "final_eval_info.json")
            rows.append(dict(item=f"{name} · final generation", time_min=round(r["gen_s_per_answer"].sum() / 60, 2),
                             peak_gpu_gb=info["peak_gpu_gb"], trainable_params=0,
                             notes=f"mean {1000*r['gen_s_per_answer'].mean():.1f} ms/answer (batch {GEN_BATCH})"))
    t = pd.DataFrame(rows); t.to_csv(TAB_DIR / "resources.csv", index=False)
    return t

def run_analysis():
    for name, nv in RUN_LIST:
        if not (RUNS_DIR / name / "TRAIN_DONE").exists():
            continue
        plot_curves(name)
        rf = RUNS_DIR / name / "final_test_records.json"
        if rf.exists():
            ch = qualitative(name, jload(rf), SEL)
            print(f"[{name}] qualitative examples:", [c[3] for c in ch])
    print(resources_table().to_string(index=False))
    if (TAB_DIR / "final_vis49.csv").exists() and (TAB_DIR / "final_vis196.csv").exists():
        cmp_ = pd.concat([pd.read_csv(TAB_DIR / "final_vis196.csv"), pd.read_csv(TAB_DIR / "final_vis49.csv")])
        cmp_.to_csv(TAB_DIR / "compression_comparison.csv", index=False)
        print(cmp_[["run", "suite", "condition", "accuracy", "pair_accuracy", "mean_ref_nll"]].round(3).to_string(index=False))
    print("tables →", TAB_DIR, "| figures →", FIG_DIR)

In [15]:
if RUN_ANALYSIS:
    run_analysis()

[vis196] qualitative examples: ['gr_test_counting_035', 'gr_test_counting_035', 'gr_test_counting_032', 'gr_test_presence_000', 'gr_test_presence_000', 'gr_test_presence_002', 'vqa_val_156754005_554749001', 'vqa_val_156754005_554749001', 'vqa_val_162581009_563575006', 'vqa_val_124949006_181303002', 'vqa_val_124949006_181303002', 'vqa_val_109537002_567011000', 'vqa_val_129576010_368980006', 'vqa_val_129576010_368980006', 'vqa_val_111644000_404820002']
                               item  time_min  peak_gpu_gb  trainable_params                                                                         notes
feature caching (download + SigLIP)      8.90        1.365                 0                                              9283 images, 2.603 GB FP16 cache
            vis196 · stage1_caption     46.50        1.573           1837056                                              mixture draws {'caption': 18000}
           vis196 · stage2_instruct     35.30        1.567           6424576    